In [6]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))
from notebooks.preprocessing import CreditPreprocessor

In [7]:
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
xgb_model = joblib.load(MODELS_DIR / "xgb_credit_model.pkl")
scaler = joblib.load(MODELS_DIR / "scaler.pkl")
preprocessor = CreditPreprocessor.load(MODELS_DIR / "preprocessor.json")

raw_data = pd.read_csv(DATA_DIR / "test.csv")

In [9]:
test_data = preprocessor.transform(raw_data)

In [10]:
test_data.isnull().sum()

RevolvingUtilizationOfUnsecuredLines               0
age                                                0
NumberOfTime30-59DaysPastDueNotWorse               0
DebtRatio                                          0
MonthlyIncome                                      0
NumberOfOpenCreditLinesAndLoans                    0
NumberOfTimes90DaysLate                            0
NumberRealEstateLoansOrLines                       0
NumberOfTime60-89DaysPastDueNotWorse               0
NumberOfDependents                                 0
MonthlyIncomeMissing                               0
NumberOfDependentsMissing                          0
NumberOfTime30-59DaysPastDueNotWorseInvalidFlag    0
NumberOfTime60-89DaysPastDueNotWorseInvalidFlag    0
NumberOfTimes90DaysLateInvalidFlag                 0
TotalMonthlyDebt                                   0
IncomePerDependentAdjusted                         0
TotalDelinquencies                                 0
RealEstateToTotalCredit                       

In [11]:
print("Are columns matching exactly?", list(test_data.columns) == list(scaler.feature_names_in_))

Are columns matching exactly? True


In [12]:
test_data_scaled = scaler.transform(test_data)

test_data_proba = xgb_model.predict_proba(test_data_scaled)[:,1]

print(f"Min    : {test_data_proba.min():.4f}")
print(f"Max    : {test_data_proba.max():.4f}")
print(f"Mean   : {test_data_proba.mean():.4f}")
print(f"Median : {np.median(test_data_proba):.4f}")



Min    : 0.0051
Max    : 0.9907
Mean   : 0.2967
Median : 0.2051


In [13]:
check_df = raw_data.copy()
check_df['Probability'] = test_data_proba
display(check_df.sort_values(by='Probability', ascending=False).head(10)[['Id', 'MonthlyIncome', 'DebtRatio', 'Probability']])

,Id,MonthlyIncome,DebtRatio,Probability
34318,113526,NaN,1776.000000,0.990653
38727,128339,4166.0,0.632109,0.990095
25464,84699,3770.0,1.282418,0.990045
30490,101127,NaN,1970.000000,0.989502
1220,3929,6823.0,0.612691,0.985697
8028,26554,4600.0,0.174093,0.984936
41626,138017,NaN,974.000000,0.984806
18856,62640,NaN,921.000000,0.984555
13227,43948,3878.0,0.732921,0.984453
5443,18144,NaN,960.000000,0.984357


In [14]:
original_test = pd.read_csv(DATA_DIR /"test.csv")

submission = pd.DataFrame()
if 'Id' in original_test.columns:
    submission['Id'] = original_test['Id']
else:
    submission['Id'] = range(len(test_data_proba))

submission['Probability'] = test_data_proba


submission.to_csv(DATA_DIR / "xgb_submission.csv", index=False)

display(submission.head())

,Id,Probability
0,0,0.857220
1,9,0.197328
2,17,0.160479
3,18,0.319921
4,24,0.146525
